# GFlowNet for Sequence Generation (GPU Accelerated)

GPU-accelerated batch training using `train_fast()`. Samples `batch_size` trajectories in parallel.

$P(x) \propto R(x)$

In [ ]:
import time
import torch
import matplotlib.pyplot as plt

from gfn import (
    train_fast,
    FastTrainingConfig,
    train,
    TrainingConfig,
    generate_greedy_trajectory,
    EnvConfig,
    set_env_config,
    print_env_info,
)
from gfn.reward import TargetMatchReward
from gfn.visualization import (
    plot_training_curves,
    plot_reward_distribution,
    plot_state_space,
    plot_flow_network,
    print_policy,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

%matplotlib inline

## 1. Configuration

In [ ]:
rna_short = EnvConfig(alphabet=['A', 'U', 'G', 'C'], max_seq_len=4)
set_env_config(rna_short)
print_env_info()

In [ ]:
target_sequences = [
    ['A', 'U', 'U', 'C'],  # AUUC
    ['A', 'U', 'C', 'C'],  # AUCC
    ['C', 'A', 'C', 'C'],  # CACC
    ['C', 'U', 'A', 'ε'],  # CUA
    ['C', 'C', 'U', 'A'],  # CCUA
    ['C', 'C', 'C', 'A'],  # CCCA
    ['G', 'G', 'G', 'G'],  # Poly-G
    ['A', 'G', 'A', 'A'],  # AGAA
    ['A', 'C', 'G', 'G'],  # ACGG
    ['A', 'A', 'G', 'A'],  # AAGA
    ['U', 'G', 'C', 'C'],  # UGCC
    ['A', 'G', 'G', 'A'],  # AGGA
    ['U', 'U', 'U', 'C'],  # UUUC
]

reward_fn = TargetMatchReward(target_sequences, r_min=0.1)

config = FastTrainingConfig(
    seed=43,
    hidden_layers=32,
    batch_size=256,
    n_iterations=500,
    learning_rate=3e-3,
    device='cuda',
    target_sequences=target_sequences,
    max_seq_len=4,
)

## 2. Training (GPU Accelerated)

In [ ]:
start_time = time.time()
result = train_fast(reward_fn, config, verbose=True)
gpu_time = time.time() - start_time

print(f"\nTraining completed in {gpu_time:.2f}s")
print(f"Final Z = {result.final_Z:.2f}")

## 3. Training Results

In [ ]:
fig = plot_training_curves(result)
plt.suptitle("GPU Fast Training Curves", y=1.02)
plt.show()

print(f"\nFinal Z = {result.final_Z:.2f}")

In [ ]:
plot_reward_distribution(result, reward_fn)

## 4. Analysis

In [ ]:
trajectory = generate_greedy_trajectory(result.model)

for t, state in enumerate(trajectory):
    seq = ''.join(s for s in state[1] if s != 'ε') or 'ε'
    print(f"  t={t}: {seq}")

In [ ]:
fig = plot_state_space(trajectory=trajectory, target_sequences=target_sequences)
plt.title("State Space with Greedy Trajectory (GPU trained)")
plt.show()

In [ ]:
plot_flow_network(
    result.model,
    target_sequences=target_sequences,
    edge_flow_threshold=0.2,
    show_nontarget_terminal_labels=False,
)
plt.show()

In [ ]:
## 5. FL-DB Training (Forward-Looking Detailed Balance)

FL-DB uses `DBModel` with per-state flow F(s) and incorporates intermediate rewards at each transition. Uses `AlignmentReward` for partial credit.

In [ ]:
from gfn.reward import AlignmentReward

target_strings = [''.join(s for s in seq if s != 'ε') for seq in target_sequences]
alignment_reward_fn = AlignmentReward(target_strings, match_score=1.0, mismatch_score=-0.5, gap_score=-0.5, r_min=0.1)

fldb_config = FastTrainingConfig(
    seed=43,
    hidden_layers=32,
    batch_size=256,
    n_iterations=500,
    learning_rate=3e-3,
    device='cuda',
    objective='FLDB',
    n_reward_workers=16,
    target_sequences=target_sequences,
    max_seq_len=4,
)

start_time = time.time()
fldb_result = train_fast(alignment_reward_fn, fldb_config, verbose=True)
fldb_time = time.time() - start_time

print(f"\nFL-DB completed in {fldb_time:.2f}s")
print(f"Final Z (log F(s0)) = {fldb_result.final_Z:.2f}")

In [ ]:
fig = plot_training_curves(fldb_result)
plt.suptitle("FL-DB Fast Training Curves", y=1.02)
plt.show()

plot_reward_distribution(fldb_result, alignment_reward_fn)

In [ ]:
print(f"{'Metric':<25} {'TB':<15} {'FL-DB':<15}")
print(f"{'Total Episodes':<25} {config.batch_size * config.n_iterations:<15,} {fldb_config.batch_size * fldb_config.n_iterations:<15,}")
print(f"{'Training Time (s)':<25} {gpu_time:<15.2f} {fldb_time:<15.2f}")
print(f"{'Final Z':<25} {result.final_Z:<15.2f} {fldb_result.final_Z:<15.2f}")
print(f"{'Final Loss':<25} {result.losses[-1]:<15.4f} {fldb_result.losses[-1]:<15.4f}")

tb_hr = f"{result.final_hit_rate*100:.2f}%" if result.hit_rates else "N/A"
fldb_hr = f"{fldb_result.final_hit_rate*100:.2f}%" if fldb_result.hit_rates else "N/A"
print(f"{'Final Hit Rate':<25} {tb_hr:<15} {fldb_hr:<15}")

In [ ]:
fldb_trajectory = generate_greedy_trajectory(fldb_result.model)
for t, state in enumerate(fldb_trajectory):
    seq = ''.join(s for s in state[1] if s != 'ε') or 'ε'
    print(f"  t={t}: {seq}")

In [ ]:
fig = plot_state_space(trajectory=fldb_trajectory, target_sequences=target_sequences)
plt.title("State Space with Greedy Trajectory (FL-DB)")
plt.show()

In [ ]:
plot_flow_network(
    fldb_result.model,
    target_sequences=target_sequences,
    edge_flow_threshold=0.2,
    show_nontarget_terminal_labels=False,
)
plt.title("Flow Network (FL-DB)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax1 = axes[0]
ax1.plot(result.losses, label='TB', alpha=0.8)
ax1.plot(fldb_result.losses, label='FL-DB', alpha=0.8)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(result.logZs, label='TB (logZ)', alpha=0.8)
ax2.plot(fldb_result.logZs, label='FL-DB (log F(s0))', alpha=0.8)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Log Partition Function')
ax2.set_title('Log Z Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
if result.hit_rates is not None:
    ax3.plot(result.hit_rates, label='TB', alpha=0.8)
if fldb_result.hit_rates is not None:
    ax3.plot(fldb_result.hit_rates, label='FL-DB', alpha=0.8)
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Target Hit Rate')
ax3.set_title('Hit Rate Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"Final Target Hit Rates:")
print(f"  TB:    {result.final_hit_rate*100:.2f}%" if result.hit_rates else "  TB:    N/A")
print(f"  FL-DB: {fldb_result.final_hit_rate*100:.2f}%" if fldb_result.hit_rates else "  FL-DB: N/A")